# Notebook 4: ML Forecasting & Anomaly Detection

This notebook tests whether the teleconnection signals identified in Notebook 3 translate into operational forecast skill for compound wind hydro energy droughts.

**Sub question:** Which forecasting paradigm is most appropriate for climate teleconnection prediction: regularised linear methods, tree based ensembles, boosted ensembles, or sequence models?

**Structure:**
1. Feature engineering based on Notebook 3 findings
2. Walk forward cross validation with nested hyperparameter tuning
3. Four model comparison (ElasticNet, Random Forest, XGBoost, LSTM)
4. Feature group comparison (climate only vs local only vs combined)
5. Feature importance analysis
6. Anomaly detection

**Key inputs from Notebook 3:**
- SAM is the primary compound driver (DJF/MAM, lags 1-4)
- IOD is the strongest individual teleconnection (JJA runoff, lags 2-4 and 8-11)
- ONI is hydro-specific (JJA/SON, lags 0-3)
- Season is essential context (SAM reverses sign between DJF and JJA)
- SAM-WHDI teleconnection strength varies on decadal timescales

## Imports and Data Loading

In [3]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Models
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv(
    "../data/processed/whdi_timeseries.csv", index_col="date", parse_dates=True
)

with open("../results/tables/feature_decisions.json", "r") as f:
    feature_decisions = json.load(f)

print(f"Data loaded: {df.shape}")
print(f"Period: {df.index[0]} to {df.index[-1]}")
print(
    f"\nPrimary target: {feature_decisions['primary_target']['variable']} "
    f"at {feature_decisions['primary_target']['lead_time']}-month lead"
)
print(
    f"Secondary target: {feature_decisions['secondary_target']['variable']} "
    f"at {feature_decisions['secondary_target']['lead_time']}-month lead"
)

Data loaded: (564, 13)
Period: 1979-01-01 00:00:00 to 2025-12-01 00:00:00

Primary target: whdi_3 at 3-month lead
Secondary target: whdi_6 at 3-month lead
